# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, following the standard Croissant schema approach.

### Dataset Source
FAIR^2 Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and initialize the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id` values in the dataset.

In [ ]:
# Display available record sets and fields with their @id
from typing import List

if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
    print(f"Total record sets found: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"Record set: {rs.name}, @id: {rs.id}")
        print("  Fields:")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    - {getattr(field, 'name', '(unnamed)')}, @id: {getattr(field, 'id', '(no id)')}")
        print()
else:
    print("No record sets found in dataset metadata.")

# Store record set @ids for later usage
record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis.

_**Note**: We use each record set's `@id` for referencing, as per best practice._

In [ ]:
# Extract data from all record sets using their @ids
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}")

# List all DataFrame columns for each record set
for record_set_id, df in dataframes.items():
    print(f"\nColumns for record set {record_set_id}: {df.columns.tolist()}")

# As an example, show the head() of the first non-empty DataFrame
for record_set_id, df in dataframes.items():
    if not df.empty:
        print(f"\nExample records from record set {record_set_id}:")
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes.

---
**Note:** _All entities (record sets, fields, columns) are referenced using their `@id` values, as above._

In [ ]:
# Example exploratory analysis for a record set with numeric fields
import numpy as np

# Select a record set and a numeric field (reference by @id) for demonstration
example_record_set_id = None
numeric_field_id = None
group_field_id = None

# Heuristic: pick the first record set with a numeric field
for rs in getattr(metadata, 'record_sets', []):
    for field in getattr(rs, 'fields', []):
        # Try to find a known numeric field type
        if hasattr(field, 'data_type') and field.data_type in ('schema:Integer', 'schema:Number', 'schema:Float'):
            example_record_set_id = rs.id
            numeric_field_id = field.id
            # Try to find a potential grouping (categorical) field as well
            for f2 in rs.fields:
                if f2.id != numeric_field_id and hasattr(f2, 'data_type') and field.data_type in ('schema:Text', 'schema:Boolean'):
                    group_field_id = f2.id
                    break
            break
    if example_record_set_id:
        break

if example_record_set_id and numeric_field_id:
    df = dataframes[example_record_set_id]
    print(f"Sample data from record set {example_record_set_id} (numeric field: {numeric_field_id}):")
    display(df[[col for col in df.columns if numeric_field_id in col or (group_field_id and group_field_id in col)]].head())
    # Ensure numeric type
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records in {example_record_set_id} with {numeric_field_id} > mean (i.e. {threshold:.2f}):")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# If previous EDA step selected valid field ids, plot histogram/boxplot
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id} in record set {example_record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
            plt.title(f"{numeric_field_id} grouped by {group_field_id}")
            plt.xticks(rotation=30)
            plt.show()
else:
    print("No numeric field identified for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and inspect the FAIR^2 dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the `mlcroissant` Python library.

- The notebook loaded dataset metadata and explored available record sets and fields (always referenced by their `@id`).
- Data from each record set was loaded for processing and basic exploratory data analysis.
- Numeric fields were used to illustrate record filtering, normalization, grouping, and visualization.

You can extend this workflow for your own downstream analyses by referencing fields and record sets by their `@id` and applying more advanced statistical or machine learning methods as needed.